# Phase 5: Exploratory Data Analysis

Revenue trends, day-of-week and monthly patterns, transaction value distribution, and top products/customers by revenue.

## Setup

In [1]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

df = pd.read_csv('../data/polokwane_sales_clean.csv', parse_dates=['date'])
sales = df[df['value_zar'] > 0].copy()   # genuine sales lines for revenue-based EDA
sales['year'] = sales['date'].dt.year
sales['month'] = sales['date'].dt.to_period('M').astype(str)
sales['dow'] = sales['date'].dt.day_name()

out = '../outputs'

## Monthly revenue trend

In [2]:
monthly = sales.groupby('month')['value_zar'].sum()
fig, ax = plt.subplots(figsize=(11,4.5))
monthly.plot(ax=ax, color='#2563eb')
ax.set_title('Monthly Revenue Trend')
ax.set_ylabel('Revenue (ZAR)')
ax.set_xlabel('Month')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(f'{out}/monthly_revenue_trend.png', dpi=150)
plt.close()

## Revenue by day of week

In [3]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_rev = sales.groupby('dow')['value_zar'].sum().reindex(dow_order)
fig, ax = plt.subplots(figsize=(7,4.5))
dow_rev.plot(kind='bar', ax=ax, color='#2563eb')
ax.set_title('Revenue by Day of Week')
ax.set_ylabel('Revenue (ZAR)')
plt.tight_layout()
plt.savefig(f'{out}/revenue_by_dow.png', dpi=150)
plt.close()

## Transaction value distribution

In [4]:
fig, ax = plt.subplots(figsize=(7,4.5))
sales[sales['value_zar'] < sales['value_zar'].quantile(0.99)]['value_zar'].hist(bins=60, ax=ax, color='#2563eb')
ax.set_title('Transaction Value Distribution (below 99th pct)')
ax.set_xlabel('Value (ZAR)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.savefig(f'{out}/value_distribution.png', dpi=150)
plt.close()

## Top 15 products by revenue

In [5]:
top_products = sales.groupby('product_desc')['value_zar'].sum().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8,6))
top_products[::-1].plot(kind='barh', ax=ax, color='#2563eb')
ax.set_title('Top 15 Products by Revenue')
ax.set_xlabel('Revenue (ZAR)')
plt.tight_layout()
plt.savefig(f'{out}/top15_products.png', dpi=150)
plt.close()

## Top 15 customers by revenue

In [6]:
top_customers = sales.groupby('debtor_name')['value_zar'].sum().sort_values(ascending=False).head(15)
fig, ax = plt.subplots(figsize=(8,6))
top_customers[::-1].plot(kind='barh', ax=ax, color='#2563eb')
ax.set_title('Top 15 Customers by Revenue')
ax.set_xlabel('Revenue (ZAR)')
plt.tight_layout()
plt.savefig(f'{out}/top15_customers.png', dpi=150)
plt.close()

## Yearly summary table

In [7]:
yearly = sales.groupby('year').agg(revenue=('value_zar','sum'), transactions=('value_zar','count'),
                                     avg_txn_value=('value_zar','mean')).round(2)
yearly.to_csv(f'{out}/yearly_summary.csv')

summary = {
    'total_revenue': float(sales['value_zar'].sum()),
    'total_transactions': int(len(sales)),
    'avg_transaction_value': float(sales['value_zar'].mean()),
    'median_transaction_value': float(sales['value_zar'].median()),
    'best_month': monthly.idxmax(), 'best_month_revenue': float(monthly.max()),
    'worst_month': monthly.idxmin(), 'worst_month_revenue': float(monthly.min()),
    'best_dow': dow_rev.idxmax(), 'worst_dow': dow_rev.idxmin(),
    'top_product': top_products.index[0], 'top_product_revenue': float(top_products.iloc[0]),
    'top_customer': top_customers.index[0], 'top_customer_revenue': float(top_customers.iloc[0]),
}
import json
with open(f'{out}/eda_summary.json','w') as f:
    json.dump(summary, f, indent=2, default=str)

print(json.dumps(summary, indent=2, default=str))
print("\nYearly summary:")
print(yearly)

{
  "total_revenue": 98282583.78,
  "total_transactions": 535920,
  "avg_transaction_value": 183.3904011419615,
  "median_transaction_value": 69.57,
  "best_month": "2025-12",
  "best_month_revenue": 3365136.3,
  "worst_month": "2026-08",
  "worst_month_revenue": 1931004.16,
  "best_dow": "Friday",
  "worst_dow": "Sunday",
  "top_product": "Shoulder Ribs Smoked P/Kg",
  "top_product_revenue": 5059321.67,
  "top_customer": "Polokwane Cash Account - Butchery",
  "top_customer_revenue": 87966777.69
}

Yearly summary:
          revenue  transactions  avg_txn_value
year                                          
2023  16461915.77         87509         188.12
2024  29745136.98        157255         189.15
2025  31761387.15        171808         184.87
2026  20314143.88        119348         170.21
